# 12 — Final dataset for criteria analysis

Build the final list of works that feed the criteria-extraction pipeline.
Applies the same filters as notebook 11:

1. `selected_english_translation == 1`
2. `historian == 0` (non-historian authors only)
3. Exclude `Unknown` and `Pseudo-Plutarch`
4. `author_impact_date` must be present
5. Drop works scored `factuality == 1` (mythic / speculative) from
   `works_factuality_v18.tsv`
6. Keep only works that fall inside one of the 4 analysis periods

**Output**: `data/processed_data/final_dataset_for_criteria.tsv`.

*Follows `notebook_rule.md`: data output is created at the end of the
notebook; every filter step logs its row count.*

## 1. Setup — imports, paths, constants

In [1]:
import random
from pathlib import Path

import pandas as pd

SEED = 0
random.seed(SEED)

META_TSV = Path('../data/clean/perseus/perseus_works_wikidata.tsv')
FACT_TSV = Path('../data/clean/final/works_factuality_v18.tsv')
OUTPUT_TSV = Path('../data/clean/final/final_dataset_for_criteria.tsv')

EXCLUDE_AUTHORS = {'unknown', 'pseudo-plutarch'}

PERIODS = [
    ('Classical (500–360 BCE)',                        -500, -360),
    ('Late Classical (354–165 BCE)',                   -354, -165),
    ('Hellenistic & Early Roman (165 BCE – 105 CE)',  -165,  105),
    ('High Roman Empire (135–205 CE)',                  135,  205),
]

## 2. Load data

In [2]:
meta_raw = pd.read_csv(META_TSV, sep='\t')
fact_raw = pd.read_csv(FACT_TSV, sep='\t')[['perseus_id',
                                              'factuality',
                                              'factuality_reason']]

print(f'Meta: {len(meta_raw):,} works from {META_TSV.name}')
print(f'Factuality: {len(fact_raw):,} rows from {FACT_TSV.name}')
meta_raw[['file_id', 'perseus_author', 'perseus_title',
          'author_impact_date', 'historian']].head()

Meta: 1,096 works from perseus_works_wikidata.tsv
Factuality: 391 rows from works_factuality_v18.tsv


,file_id,perseus_author,perseus_title,author_impact_date,historian
0,tlg0003.tlg001.perseus-eng6,Thucydides,History of the Peloponnesian War,-425,1
1,tlg0003.tlg001.1st1K-eng1,Thucydides,The Peloponnesian War,-425,1
2,tlg0003.tlg001.perseus-eng4,Thucydides,The History of the Grecian War,-425,1
3,tlg0003.tlg001.1st1K-ger1,Thucydides,Vier Staatsreden aus Thucydides,-425,1
4,tlg0003.tlg001.perseus-grc2,Thucydides,Ἱστορίαι,-425,1


## 3. Preprocessing — apply the six filters

In [3]:
n0 = len(meta_raw)

df = meta_raw[(meta_raw['selected_english_translation'] == 1)
              & (meta_raw['historian'] == 0)].copy()
df = df[~df['perseus_author'].astype(str).str.strip()
                              .str.lower().isin(EXCLUDE_AUTHORS)]
df['year'] = pd.to_numeric(df['author_impact_date'], errors='coerce')
df = df[df['year'].notna()].copy()
df['year'] = df['year'].astype(int)
df['n_pages'] = pd.to_numeric(df['n_pages'], errors='coerce').fillna(0).astype(int)
n1 = len(df)

df = df.merge(fact_raw, on='perseus_id', how='left')
df = df[df['factuality'] != 1].copy()
n2 = len(df)


def assign_period(year):
    for label, lo, hi in PERIODS:
        if lo <= year <= hi:
            return label
    return None


df['period'] = df['year'].apply(assign_period)
df = df[df['period'].notna()].copy()
n3 = len(df)

assert df['file_id'].is_unique
assert df['period'].notna().all()

print(f'Raw:                              {n0:,}')
print(f'After base filters:               {n1:,}  (dropped {n0-n1})')
print(f'After factuality != 1 (mythic):   {n2:,}  (dropped {n1-n2})')
print(f'After period assignment:          {n3:,}  (dropped {n2-n3})')
print(df['period'].value_counts())
df.head()

Raw:                              1,096
After base filters:               401  (dropped 695)
After factuality != 1 (mythic):   328  (dropped 73)
After period assignment:          328  (dropped 0)
period
Classical (500–360 BCE)                         155
Late Classical (354–165 BCE)                     80
High Roman Empire (135–205 CE)                   58
Hellenistic & Early Roman (165 BCE – 105 CE)     35
Name: count, dtype: int64


,file_id,perseus_id,perseus_author,wikidata_work_id,wikidata_work_label,selected_english_translation,historian,main_language,author_code,author_wikidata_id,...,genre,form_of_creative_work,confidence,polity_group,keep_greek_focus,is_scientific,year,factuality,factuality_reason,period
19,tlg0008.tlg001.perseus-eng2,tlg0008.tlg001,Athenaeus,Q1244416,Deipnosophistae,1,0,English,tlg0008,Q294923,...,NaN,anthology,93,NaN,NaN,0,205,3.0,Athenaeus' 'Deipnosophistae' is an anthology o...,High Roman Empire (135–205 CE)
20,tlg0010.tlg009.perseus-eng2,tlg0010.tlg009,Isocrates,Q87746371,Helen,1,0,English,tlg0010,Q221182,...,NaN,NaN,100,NaN,NaN,0,-400,4.0,"Isocrates' 'Helen' is a forensic oration, whic...",Classical (500–360 BCE)
21,tlg0010.tlg007.perseus-eng2,tlg0010.tlg007,Isocrates,Q110733998,To Demonicus,1,0,English,tlg0010,Q221182,...,NaN,NaN,100,NaN,NaN,0,-400,4.0,Isocrates' 'To Demonicus' is a forensic oratio...,Classical (500–360 BCE)
22,tlg0010.tlg001.perseus-eng2,tlg0010.tlg001,Isocrates,Q110734275,Against Euthynus,1,0,English,tlg0010,Q221182,...,NaN,NaN,100,NaN,NaN,0,-400,4.0,Isocrates' 'Against Euthynus' is a forensic or...,Classical (500–360 BCE)
23,tlg0010.tlg006.perseus-eng2,tlg0010.tlg006,Isocrates,Q16329966,Aegineticus,1,0,English,tlg0010,Q221182,...,NaN,NaN,100,NaN,NaN,0,-400,4.0,Isocrates' 'Aegineticus' is a forensic oration...,Classical (500–360 BCE)


## 4. Summary — period × author/work/page totals

In [4]:
summary = (df.groupby('period')
             .agg(n_authors=('perseus_author', 'nunique'),
                  n_works=('file_id', 'count'),
                  total_pages=('n_pages', 'sum'),
                  total_words=('n_words', 'sum'))
             .reset_index())
summary.loc['TOTAL'] = [
    'TOTAL',
    df['perseus_author'].nunique(),
    len(df),
    int(df['n_pages'].sum()),
    int(df['n_words'].sum()) if 'n_words' in df.columns else 0,
]
summary

,period,n_authors,n_works,total_pages,total_words
0,Classical (500–360 BCE),9,155,5146,1287989
1,Hellenistic & Early Roman (165 BCE – 105 CE),6,35,1385,347221
2,High Roman Empire (135–205 CE),7,58,4920,1230267
3,Late Classical (354–165 BCE),7,80,5089,1272165
TOTAL,TOTAL,29,328,16540,4137642


## 5. Author breakdown per period

In [5]:
author_breakdown = (df.groupby(['period', 'perseus_author'])
                      .agg(n_works=('file_id', 'count'),
                           total_pages=('n_pages', 'sum'))
                      .reset_index()
                      .sort_values(['period', 'total_pages'],
                                   ascending=[True, False]))
author_breakdown

,period,perseus_author,n_works,total_pages
8,Classical (500–360 BCE),Plato,33,2121
6,Classical (500–360 BCE),Isocrates,30,819
2,Classical (500–360 BCE),Aristophanes,11,713
3,Classical (500–360 BCE),Hippocrates,19,597
7,Classical (500–360 BCE),Lysias,34,349
5,Classical (500–360 BCE),Isaeus,12,208
1,Classical (500–360 BCE),Antiphon,6,135
0,Classical (500–360 BCE),Andocides,4,128
4,Classical (500–360 BCE),Hyperides,6,76
13,Hellenistic & Early Roman (165 BCE – 105 CE),New Testament,27,699


## 6. Save the final dataset

In [6]:
output_cols = [
    'file_id', 'perseus_id', 'perseus_author', 'perseus_title',
    'wikidata_work_id', 'wikidata_work_label',
    'author_wikidata_id', 'author_impact_date', 'year', 'period',
    'genre', 'form_of_creative_work', 'instance_of',
    'factuality', 'factuality_reason',
    'main_language', 'languages', 'editors', 'pub_date',
    'n_characters', 'n_words', 'n_pages', 'file_path',
]
output_cols = [c for c in output_cols if c in df.columns]

out = (df[output_cols]
        .sort_values(['period', 'year', 'perseus_author', 'perseus_title'])
        .reset_index(drop=True))

OUTPUT_TSV.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_TSV, sep='\t', index=False)

print(f'Saved {len(out)} works → {OUTPUT_TSV}')
out.head()

Saved 328 works → ../data/clean/final/final_dataset_for_criteria.tsv


,file_id,perseus_id,perseus_author,perseus_title,wikidata_work_id,wikidata_work_label,author_wikidata_id,author_impact_date,year,period,...,factuality,factuality_reason,main_language,languages,editors,pub_date,n_characters,n_words,n_pages,file_path
0,tlg0027.tlg004.perseus-eng2,tlg0027.tlg004,Andocides,Against Alcibiades,NaN,NaN,Q492228,-500,-500,Classical (500–360 BCE),...,4.0,"Andocides was a forensic orator, and his works...",English,eng; grc; lat,Kenneth John Maidment,1941,29491,5226,21,tlg0027/tlg004/tlg0027.tlg004.perseus-eng2.xml
1,tlg0027.tlg002.perseus-eng2,tlg0027.tlg002,Andocides,On His Return,Q27663867,On His Return,Q492228,-500,-500,Classical (500–360 BCE),...,4.0,"Andocides was a forensic orator, and his works...",English,grc; eng; lat,Kenneth John Maidment,1941,16263,2941,12,tlg0027/tlg002/tlg0027.tlg002.perseus-eng2.xml
2,tlg0027.tlg001.perseus-eng2,tlg0027.tlg001,Andocides,On the Mysteries,Q87737021,On the Mysteries,Q492228,-500,-500,Classical (500–360 BCE),...,4.0,"Andocides was a forensic orator, and his works...",English,grc; eng; lat,Kenneth John Maidment,1941,106628,18619,74,tlg0027/tlg001/tlg0027.tlg001.perseus-eng2.xml
3,tlg0027.tlg003.perseus-eng2,tlg0027.tlg003,Andocides,On the Peace with Sparta,Q87737023,On the Peace with Sparta,Q492228,-500,-500,Classical (500–360 BCE),...,4.0,"Andocides was a forensic orator, and his works...",English,eng; grc; lat,Kenneth John Maidment,1941,30121,5221,21,tlg0027/tlg003/tlg0027.tlg003.perseus-eng2.xml
4,tlg0028.tlg006.perseus-eng2,tlg0028.tlg006,Antiphon,On the Choreutes,Q87738364,On the Choreutes,Q15078686,-500,-500,Classical (500–360 BCE),...,4.0,"Antiphon was a forensic orator, and his works ...",English,grc; lat; eng,Kenneth John Maidment,1941,35112,6061,24,tlg0028/tlg006/tlg0028.tlg006.perseus-eng2.xml
